# M1 — gesture classifierThree classes: `none`, `flag` (single flick), `approve` (double flick).Everything here is sized from measurement rather than assumption, and severalof the numbers started out wrong. The ones that matter:| | measured | consequence ||---|---|---|| gesture duration | 480–1395 ms | window is 2.0 s, not 1.5 || receptive field needed | ≥ 1395 ms | layer 3 kernel is 7, not 3 || what separates the classes | oscillation count (2 vs 5) | **not** duration, which overlaps and even inverts || execution drift between sessions | ~50% | time-warp augmentation is ±40% || clipping | 3.3% of samples | amplitude saturates; shape must carry it |**The bar to beat is 87.1%** — a three-feature threshold rule. A model thatcan't clear that has learned nothing a handful of `if` statements couldn't.

In [ ]:
import numpy as np, torch, torch.nn as nn, torch.nn.functional as Fimport matplotlib.pyplot as plttorch.manual_seed(0); np.random.seed(0)DEV = "mps" if torch.backends.mps.is_available() else "cpu"d = np.load("../data/windows.npz", allow_pickle=True)X, y = d["X"], d["y"]LABELS = [str(s) for s in d["labels"]]SESSION = d["session"]print("X", X.shape, "  y", y.shape, "  device", DEV)for i, name in enumerate(LABELS):    print(f"  {name:<9} {(y == i).sum():5d}")

## 1. Look at the data firstThree channels, 50 samples, in g with gravity removed. Never train on data youhave not plotted — a mislabelled class or a dead axis is obvious here andinvisible in a loss curve.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(13, 6), sharex=True, sharey=True)for row, name in enumerate(LABELS):    idx = np.where(y == row)[0][:3]    for col, i in enumerate(idx):        ax = axes[row, col]        for ch, c in zip(X[i], ("tab:red", "tab:green", "tab:blue")):            ax.plot(ch, color=c, lw=1)        ax.set_title(f"{name}  #{i}", fontsize=9)        ax.grid(alpha=.3)axes[0, 0].set_ylabel("g"); axes[2, 1].set_xlabel("sample (25 Hz)")plt.tight_layout(); plt.show()

`approve` should show a longer oscillation train than `flag` — that is thesignal. If they look identical, stop here: the labels are wrong, not the model.

## 2. The baseline to beatThree hand-written features: peak amplitude, active duration, oscillation count.

In [ ]:
import sys; sys.path.insert(0, "..")from whip import baselineswins = [x.tolist() for x in X]truth = [LABELS[i] for i in y]r = baselines.evaluate(wins, truth)print(f"threshold baseline: {r['accuracy']*100:.1f}%")for k, v in r["per_class"].items():    print(f"  {k:<9} {v*100:5.1f}%")BASELINE = r["accuracy"]

## 3. Split by session — and, with one session, by timeWindows overlap 88% (50 samples, stride 6) and one gesture yields ~6 of them.A random split puts near-identical windows in train and test and reports ~98%that means nothing.**With only one prompted session there is no session-level holdout for thepositive classes.** Holding out negative sessions alone leaves the test set with*zero* positives — accuracy then measures only how badly negatives are rejected,and gesture recall is unmeasurable.So: hold out the **last 30% of the prompted session by time**, plus two negativesessions. A temporal split still shares session artifacts (ring rotation, biasdrift), so it is optimistic — but every class is represented and no windowstraddles the boundary.**A second session on another day is worth more than more gestures on thisone.** It is what turns this optimistic number into an honest one.

In [ ]:
sessions = sorted(set(SESSION.tolist()))for s in sessions:    m = SESSION == s    counts = {LABELS[i]: int(((y == i) & m).sum()) for i in range(3)}    print(f"  {s:<44} {counts}")START = d["start_s"]PROMPTED = "prompted_20260909_160303"NEG_TEST = {"negative_20260908_202143", "negative_20260908_201610"}prompted = SESSION == PROMPTEDcutoff = np.quantile(START[prompted], 0.70)test_mask = (prompted & (START >= cutoff)) | np.isin(SESSION, list(NEG_TEST))Xtr, ytr = X[~test_mask], y[~test_mask]Xte, yte = X[test_mask], y[test_mask]print(f"\ntemporal cutoff at {cutoff:.0f}s of the prompted session")print(f"train {len(ytr):5d}   " + str({LABELS[i]: int((ytr == i).sum()) for i in range(3)}))print(f"test  {len(yte):5d}   " + str({LABELS[i]: int((yte == i).sum()) for i in range(3)}))

## 4. The model```Conv1d(3→16,  k=5) → BN → ReLU → MaxPool2      (16, 25)Conv1d(16→32, k=5) → BN → ReLU → MaxPool2      (32, 12)Conv1d(32→64, k=7) → BN → ReLU                 (64, 12)GlobalAvg ⊕ GlobalMax → Dropout → Linear(128→3)```Two design points worth understanding:**Kernel 7 in layer 3** gives a 40-sample (1600 ms) receptive field, so one unitsees an entire gesture. At kernel 3 it was 960 ms and 74% of real gesturesexceeded it — no unit could count the oscillations that distinguish the classes.**Both poolings.** Max reports *did the multi-peak template fire* (patternidentity); average reports *how much total activity* (energy). Alone, neitherseparates "one hard flick" from "two soft ones" — together they do, because theclassifier reads the ratio.No softmax: `CrossEntropyLoss` wants logits.

In [ ]:
class GestureCNN(nn.Module):    def __init__(self, n_classes=3):        super().__init__()        self.b1 = nn.Sequential(nn.Conv1d(3, 16, 5, padding=2), nn.BatchNorm1d(16), nn.ReLU(), nn.MaxPool1d(2))        self.b2 = nn.Sequential(nn.Conv1d(16, 32, 5, padding=2), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2))        self.b3 = nn.Sequential(nn.Conv1d(32, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU())        self.drop = nn.Dropout(0.3)        self.head = nn.Linear(128, n_classes)    def forward(self, x):        x = self.b3(self.b2(self.b1(x)))        x = torch.cat([x.mean(dim=2), x.amax(dim=2)], dim=1)        return self.head(self.drop(x))model = GestureCNN()print(model)print(f"\nparameters: {sum(p.numel() for p in model.parameters()):,}  (ceiling 50,000)")

## 5. Augmentation — measured, not assumedEvery augmentation here was tested against a **cross-session** holdout, and theresult reversed the design I argued for.| augmentation | cross-session macro F1 ||---|---|| amplitude ±20% + rotation ±10° | 64.8% ± 7.8 || amplitude ±10% | 63.8% ± 11.3 || none | 62.4% ± 10.6 || rotation ±30° | 60.8% || time warp ±10% | 62.8% || time warp ±40% | 59.4% || **rot30 + amp30 + warp40 (original plan)** | **54.3%** |**Time warp is harmful at any level**, and it was the most confidently arguedchoice. The reasoning was that duration is a session-correlated shortcut, sowarping breaks it. But the real discriminator is *oscillation count*, detected assharp local maxima — and resampling through linear interpolation smooths exactlythose peaks. It destroyed the signal in order to suppress a confound.**Rotation hurts at ±30° but not ±10°**, which suggests the finger axis may notbe x as assumed. Worth checking before trusting rotation at all.Note the ±8–11% spread across seeds: at this noise level the light settings areindistinguishable from each other. Only the heavy combination is clearly worse.That variance is itself the finding — it is what too little data looks like.

In [ ]:
def augment(batch):    """    Deliberately light. Heavier augmentation measured 10 points worse    cross-session; see the table above.    """    B = batch.shape[0]    out = batch.clone()    # amplitude: a soft double must stay a double    out *= (0.8 + 0.4 * torch.rand(B, 1, 1, device=batch.device))    # small rotation about the presumed finger axis; +/-30 measurably hurt    th = (torch.rand(B, device=batch.device) * 2 - 1) * (10 * np.pi / 180)    cos, sin = torch.cos(th)[:, None], torch.sin(th)[:, None]    yy, zz = out[:, 1].clone(), out[:, 2].clone()    out[:, 1], out[:, 2] = cos * yy - sin * zz, sin * yy + cos * zz    # NO time warp: interpolation smooths the peaks that carry the signal    return out + 0.02 * torch.randn_like(out)sample = torch.tensor(X[y == 2][:1])fig, ax = plt.subplots(1, 4, figsize=(13, 2.4), sharey=True)ax[0].plot(sample[0].T); ax[0].set_title("original", fontsize=9)for i in range(1, 4):    ax[i].plot(augment(sample)[0].T); ax[i].set_title(f"augmented {i}", fontsize=9)for a in ax: a.grid(alpha=.3)plt.tight_layout(); plt.show()

## 6. TrainClass weights rather than discarding negatives — throwing away negatives throwsaway exactly the hard cases the false-positive budget depends on.

In [ ]:
def run_training(Xtr, ytr, Xte, yte, epochs=40, verbose=True):    model = GestureCNN().to(DEV)    opt = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-4)    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)    counts = np.bincount(ytr, minlength=3)    w = torch.tensor(len(ytr) / (3 * np.maximum(counts, 1)), dtype=torch.float32, device=DEV)    lossf = nn.CrossEntropyLoss(weight=w)    Xtr_t = torch.tensor(Xtr, device=DEV); ytr_t = torch.tensor(ytr, device=DEV)    Xte_t = torch.tensor(Xte, device=DEV); yte_t = torch.tensor(yte, device=DEV)    hist = []    for ep in range(epochs):        model.train()        perm = torch.randperm(len(ytr_t), device=DEV)        total = 0.0        for i in range(0, len(perm), 128):            idx = perm[i:i+128]            xb = augment(Xtr_t[idx]); yb = ytr_t[idx]            opt.zero_grad()            loss = lossf(model(xb), yb)            loss.backward(); opt.step()            total += loss.item() * len(idx)        sched.step()        model.eval()        with torch.no_grad():            acc = (model(Xte_t).argmax(1) == yte_t).float().mean().item()        hist.append((total / len(perm), acc))        if verbose and (ep + 1) % 5 == 0:            print(f"  epoch {ep+1:3d}  loss {hist[-1][0]:.4f}  test acc {acc*100:.1f}%")    return model, histmodel, hist = run_training(Xtr, ytr, Xte, yte)

In [ ]:
loss = [h[0] for h in hist]; acc = [h[1] for h in hist]fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))ax[0].plot(loss); ax[0].set_title("training loss"); ax[0].grid(alpha=.3)ax[1].plot([a*100 for a in acc], label="CNN")ax[1].axhline(BASELINE*100, color="crimson", ls="--", label=f"threshold baseline {BASELINE*100:.1f}%")ax[1].set_title("held-out accuracy"); ax[1].legend(); ax[1].grid(alpha=.3)plt.tight_layout(); plt.show()

## 7. Where it failsThe confusion matrix matters more than accuracy. `none → flag/approve` is thefalse-positive budget, and your spec puts that above F1: a classifier that fireswhile you type poisons every downstream label.

In [ ]:
model.eval()with torch.no_grad():    pred = model(torch.tensor(Xte, device=DEV)).argmax(1).cpu().numpy()cm = np.zeros((3, 3), dtype=int)for t, p in zip(yte, pred):    cm[t, p] += 1print(f"{'':<9}" + "".join(f"{n:>9}" for n in LABELS) + "   recall")for i, n in enumerate(LABELS):    seen = cm[i].sum()    rec = cm[i, i] / seen * 100 if seen else 0    print(f"{n:<9}" + "".join(f"{v:>9}" for v in cm[i]) + f"   {rec:5.1f}%")fp = cm[0, 1:].sum()if cm[0].sum():    hours = cm[0].sum() / (25 * 3600 / 6)    print(f"\nfalse fires: {fp} over ~{hours:.2f} h of negatives -> {fp/max(hours,1e-9):.1f}/hour")    print("(before debouncing; the daemon requires 4-12 consecutive windows)")

## 8. Learning curve — are we data-limited?300 per class was a guess in the spec. The curve answers it, but only if twotraps are avoided, both of which produced badly misleading curves first time:**Subsample by gesture, not by window.** Six windows per gesture at 88% overlapare near-duplicates. Varying the window count varies class balance and sessionmix rather than "amount of data".**Split validation by gesture too.** A random window split puts near-identicalcopies in train and val, so validation measures memorisation, early stoppingselects an overfitted checkpoint, and the curve *falls* as data grows. That iswhat happened: 76% at 35 gestures down to 69% at 143, which reads as "more datahurts" and is purely an artifact.Several seeds per point, because the spread is ±4-8% at this scale.

In [ ]:
import jsonfrom pathlib import Pathnotes = json.loads(Path(f"../data/sessions/{PROMPTED}.notes.json").read_text())cues = sorted((m["cue_at"], m["label"]) for m in notes["marks"])# Map each positive window to the gesture whose labelled span covers it.gesture_id = np.full(len(y), -1)for gi, (t, _) in enumerate(cues):    gesture_id[prompted & (y != 0) & (START >= t - 2.0) & (START <= t + 1.2)] = gitrain_mask = ~test_masktrain_gestures = sorted(set(gesture_id[train_mask & (gesture_id >= 0)].tolist()))print(f"{len(train_gestures)} gestures available for training\n")def macro_f1(pred, true):    out = []    for c in (1, 2):        tp = ((pred == c) & (true == c)).sum()        fp = ((pred == c) & (true != c)).sum()        fn = ((pred != c) & (true == c)).sum()        p = tp / (tp + fp) if tp + fp else 0        r = tp / (tp + fn) if tp + fn else 0        out.append(2 * p * r / (p + r) if p + r else 0)    return float(np.mean(out))rng = np.random.default_rng(0)points = []for frac in (0.25, 0.5, 0.75, 1.0):    n = max(4, int(len(train_gestures) * frac))    scores = []    for seed in range(3):        keep = list(rng.choice(train_gestures, size=n, replace=False))        mask = train_mask & ((gesture_id < 0) | np.isin(gesture_id, keep))        m, h = run_training(X[mask], y[mask], Xte, yte, epochs=30, verbose=False)        m.eval()        with torch.no_grad():            pred = m(torch.tensor(Xte, device=DEV)).argmax(1).cpu().numpy()        scores.append(macro_f1(pred, yte))    points.append((n, np.mean(scores), np.std(scores)))    print(f"  {n:4d} gestures -> {np.mean(scores)*100:5.1f}% +/- {np.std(scores)*100:.1f}")xs = [p[0] for p in points]plt.figure(figsize=(6, 3.2))plt.errorbar(xs, [p[1]*100 for p in points], yerr=[p[2]*100 for p in points],             fmt="o-", capsize=4)plt.xlabel("training gestures"); plt.ylabel("macro F1 % (flag, approve)")plt.title("still climbing => collect another session")plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 9. Export for the C++ daemonWeights plus **golden vectors** — input/output pairs the C++ must reproduce to1e-5. Every port bug I have seen is a transposed weight or an off-by-one inpadding, and both are invisible without these and instant with them.Get float parity first, quantise second. Changing both at once makes anydiscrepancy unattributable.

In [ ]:
model.eval().cpu()weights = {k: v.numpy() for k, v in model.state_dict().items()}golden_in = X[np.random.permutation(len(X))[:32]]with torch.no_grad():    golden_out = model(torch.tensor(golden_in)).numpy()np.savez("../data/model.npz", **weights, golden_in=golden_in, golden_out=golden_out)print(f"wrote model.npz: {len(weights)} tensors, {sum(v.size for v in weights.values()):,} parameters")print("golden vectors:", golden_in.shape, "->", golden_out.shape)